In [4]:
import sys
from pathlib import Path
import os

In [5]:
# Adding project root (parent of notebooks/) to PYTHONPATH
sys.path.append(str(Path.cwd().parent))

In [6]:
from ingestion.loaders import extract_all_texts

extract_all_texts("../data/resume", "../data/resume_extract")

Extracting files: 100%|██████████| 2549/2549 [00:01<00:00, 1341.52it/s]


In [7]:
assert "OPENAI_API_KEY" in os.environ, (
    "OPENAI_API_KEY is not set. "
    "Please follow the README instructions to configure it."
)

print("✅ OPENAI_API_KEY detected")


✅ OPENAI_API_KEY detected


In [10]:
from agents.structuring_agent import process_resumes
process_resumes(
    input_folder="../data/resume_extract",
    output_folder="../data/resume_extract_json"
)


Processing CVs: 100%|██████████| 2549/2549 [00:00<00:00, 3170.70CV/s]


In [11]:
import json
from pathlib import Path

sample = next(Path("../data/resume_extract_json").glob("*.json"))

with open(sample, "r", encoding="utf-8") as f:
    data = json.load(f)

data


{'name': None,
 'email': None,
 'phone': None,
 'location': 'Bat Yam',
 'summary': 'Java full stack developer with 3+ years of full-stack experience and 5+ years total software development experience, specializing in backend Java (Spring) and frontend Web technologies, with hands-on expertise in RESTful services, microservices, and database integrations (MySQL, MongoDB).',
 'skills': ['Java',
  'JavaScript',
  'HTML5',
  'CSS3',
  'jQuery',
  'Bootstrap',
  'Eclipse',
  'IntelliJ IDEA',
  'GitHub',
  'Git',
  'SQL',
  'MySQL',
  'MongoDB',
  'NoSQL',
  'OOP',
  'AOP',
  'Spring MVC',
  'JPA',
  'Hibernate',
  'JDBC',
  'Spring Boot',
  'Spring Security',
  'Spring Web',
  'REST',
  'JSON',
  'Maven',
  'JUnit',
  'Postman',
  'Linux',
  'Windows'],
 'experience': [{'title': 'Backend JAVA developer',
   'company': 'Unknown (Israel, Rehovot)',
   'start_date': '2020',
   'end_date': 'Present',
   'description': 'Developed backend RESTful web services within a microservice architecture fo

In [12]:
from ingestion.formatters import process_json_folder

process_json_folder(
    "../data/resume_extract_json",
    "../data/resume_extract_text"
)


Processing JSON files:   0%|          | 0/2549 [00:00<?, ?it/s]

Processing JSON files: 100%|██████████| 2549/2549 [01:08<00:00, 36.96it/s]


In [13]:
from pathlib import Path
sample = next(Path("../data/resume_extract_text").glob("*.txt"))
sample.read_text(encoding="utf-8")[:1000]


"Skills: Java; JavaScript; HTML5; CSS3; jQuery; Bootstrap; Eclipse; IntelliJ IDEA; GitHub; Git; SQL; MySQL; MongoDB; NoSQL; OOP; AOP; Spring MVC; JPA; Hibernate; JDBC; Spring Boot; Spring Security; Spring Web; REST; JSON; Maven; JUnit; Postman; Linux; Windows\nExperience: Backend JAVA developer at Unknown (Israel, Rehovot); Full stack JAVA developer at Bank Otkritie; Software R-Style language developer at Privatbank\nEducation: Master's degree in Computer Science and Information Technology at National Technical University of Ukraine\nSummary: Java full stack developer with 3+ years of full-stack experience and 5+ years total software development experience, specializing in backend Java (Spring) and frontend Web technologies, with hands-on expertise in RESTful services, microservices, and database integrations (MySQL, MongoDB)."

In [14]:
from ingestion.resume_index import build_resume_index

BACKEND = "faiss"   # or "chroma"

result = build_resume_index(
    input_folder="../data/resume_extract_text",
    backend=BACKEND,
    faiss_index_path="../data/resume_index.faiss",
    mapping_path="../data/resume_index_mapping.json",
    chroma_dir="../data/chroma_resume_db",
    chroma_collection="resumes",
)

result

d:\LLM\Smart_Resume_and_Job_Matcher\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Encoding resumes: 100%|██████████| 80/80 [03:58<00:00,  2.98s/it]


{'backend': 'faiss',
 'count': 2549,
 'dim': 384,
 'faiss_index_path': '../data/resume_index.faiss',
 'mapping_path': '../data/resume_index_mapping.json'}

In [16]:
from ingestion.job_index import build_jobs_index

result = build_jobs_index(
    input_folder="../data/jobs",
    faiss_index_path="../data/jobs_index.faiss",
    mapping_path="../data/jobs_index_mapping.json"
)
result


Encoding jobs: 100%|██████████| 1/1 [00:03<00:00,  3.86s/it]


{'count': 1,
 'dim': 384,
 'faiss_index_path': '../data/jobs_index.faiss',
 'mapping_path': '../data/jobs_index_mapping.json'}

In [17]:
from app.matching import search_index, read_txt

job_text = read_txt("../data/jobs/sample_job.txt")

top_resumes = search_index(
    query_text=job_text,
    index_path="../data/resume_index.faiss",
    mapping_path="../data/resume_index_mapping.json",
    top_k=10
)
top_resumes


[{'rank': 1, 'score': 0.6648896932601929, 'filename': '55.txt'},
 {'rank': 2, 'score': 0.6456311345100403, 'filename': '7.txt'},
 {'rank': 3, 'score': 0.6408270597457886, 'filename': '61.txt'},
 {'rank': 4, 'score': 0.6126968264579773, 'filename': '60.txt'},
 {'rank': 5, 'score': 0.6045936346054077, 'filename': '10.txt'},
 {'rank': 6, 'score': 0.5988060235977173, 'filename': '9.txt'},
 {'rank': 7, 'score': 0.5837372541427612, 'filename': '28.txt'},
 {'rank': 8, 'score': 0.5831563472747803, 'filename': '13.txt'},
 {'rank': 9, 'score': 0.5784345269203186, 'filename': '20.txt'},
 {'rank': 10, 'score': 0.5756862163543701, 'filename': '17.txt'}]

In [23]:
from agents.explainer_agent import explain_match_with_llm, build_llm_client
from utils.json_utils import safe_json_parse

client = build_llm_client()

def read_resume_text(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

for rank, hit in enumerate(top_resumes, start=1):
    resume_text = read_resume_text(
        filename=hit["filename"],
        base_dir="../data/resume_extract_text"
    )

    out = explain_match_with_llm(
        mode="job_to_resumes",
        similarity_score=hit["score"],
        job_text=job_text,
        resume_text=resume_text,
        top_k_rank=rank,
        client=client,
    )

    hit["llm_explanation"] = safe_json_parse(out["raw_json"])



In [24]:
print("Score:", hit["score"])
print(hit["llm_explanation"]["explanation"])
print("Strengths:")
for s in hit["llm_explanation"]["strengths"]:
    print("-", s)
print("Gaps:")
for g in hit["llm_explanation"]["gaps"]:
    print("-", g)


Score: 0.5756862163543701
This resume is relevant to the Frontend Developer position as it highlights extensive experience with React and Redux, which are essential for the role. The candidate's background in developing user interfaces and their familiarity with modern web technologies aligns well with the company's focus on creating a cutting-edge web application. Additionally, their experience working in collaborative environments suggests they can effectively engage with teams using tools like GitHub and Slack.
Strengths:
- Proven experience with React and Redux, essential for the job requirements.
- Strong background in frontend technologies and UI design, indicating a sharp eye for detail.
- Experience in collaborative environments, suggesting effective teamwork and communication skills.
Gaps:
- No mention of TypeScript, which is a key component of the tech stack.
- Lacks specific experience with Tailwind CSS and Shadcn's UI components, which are preferred technologies for the rol

In [25]:
# =========================
# MODE 1 — JOB → TOP RESUMES
# =========================

import os
from agents.explainer_agent import explain_match_with_llm, build_llm_client
from utils.json_utils import safe_json_parse

client = build_llm_client()

def read_txt(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

RESUME_DIR = "../data/resume_extract_text"  # <- your folder with resume .txt

for rank, hit in enumerate(top_resumes, start=1):
    resume_text = read_txt(hit["filename"], RESUME_DIR)

    out = explain_match_with_llm(
        mode="job_to_resumes",
        similarity_score=hit["score"],
        job_text=job_text,            # query = job
        resume_text=resume_text,      # candidate = resume
        top_k_rank=rank,
        client=client,
    )
    hit["llm_explanation"] = safe_json_parse(out["raw_json"])

    print("=" * 90)
    print(f"[JOB → RESUMES] Rank #{rank} | File: {hit['filename']} | Score: {hit['score']:.3f}\n")
    print(hit["llm_explanation"].get("explanation", ""))

    print("\nStrengths:")
    for s in hit["llm_explanation"].get("strengths", []):
        print("-", s)

    print("\nGaps:")
    for g in hit["llm_explanation"].get("gaps", []):
        print("-", g)


[JOB → RESUMES] Rank #1 | File: 55.txt | Score: 0.665

This resume is highly relevant to the Frontend Developer position as it showcases strong proficiency in React and TypeScript, which are essential for the role. The candidate has experience in developing modern web applications and a solid understanding of UI design principles, aligning well with the job's focus on well-crafted user interfaces. Additionally, their background in both frontend and backend development indicates a comprehensive understanding of the product lifecycle, which is crucial for effective collaboration with backend teams.

Strengths:
- Proficient in React and TypeScript, with practical experience in product environments.
- Strong background in UI design and development, ensuring attention to detail in user interfaces.
- Experience working in collaborative environments, which aligns with the company's emphasis on asynchronous collaboration.

Gaps:
- No direct experience mentioned with Tailwind CSS or Shadcn's UI

In [18]:
resume_text = read_txt("../data/resume_extract_text/55.txt")

top_jobs = search_index(
    query_text=resume_text,
    index_path="../data/jobs_index.faiss",
    mapping_path="../data/jobs_index_mapping.json",
    top_k=10
)
top_jobs


[{'rank': 1, 'score': 0.6648897528648376, 'filename': 'sample_job.txt'}]

In [26]:
# ======================
# MODE 2 — RESUME → TOP JOBS
# ======================

import os
from agents.explainer_agent import explain_match_with_llm, build_llm_client
from utils.json_utils import safe_json_parse

client = build_llm_client()

def read_txt(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

JOB_DIR = "../data/jobs"  # <- your folder with job .txt

for rank, hit in enumerate(top_jobs, start=1):
    job_candidate_text = read_txt(hit["filename"], JOB_DIR)

    out = explain_match_with_llm(
        mode="resume_to_jobs",
        similarity_score=hit["score"],
        resume_text=resume_text,           # query = resume
        job_text=job_candidate_text,       # candidate = job offer
        top_k_rank=rank,
        client=client,
    )
    hit["llm_explanation"] = safe_json_parse(out["raw_json"])

    print("=" * 90)
    print(f"[RESUME → JOBS] Rank #{rank} | File: {hit['filename']} | Score: {hit['score']:.3f}\n")
    print(hit["llm_explanation"].get("explanation", ""))

    print("\nStrengths:")
    for s in hit["llm_explanation"].get("strengths", []):
        print("-", s)

    print("\nGaps:")
    for g in hit["llm_explanation"].get("gaps", []):
        print("-", g)


[RESUME → JOBS] Rank #1 | File: sample_job.txt | Score: 0.665

This job offer for a Frontend Developer aligns well with the candidate's extensive experience in frontend technologies, particularly React and Redux. The emphasis on creating a modern web application and collaborating with a small team matches the candidate's background in developing and maintaining software in various environments. Additionally, the focus on UI design and user journeys resonates with the candidate's skills and passion for well-crafted user interfaces.

Strengths:
- Proven experience with React and Redux, which are key technologies in the job offer.
- Strong background in frontend development and UI design, aligning with the company's focus on well-crafted user interfaces.
- Experience working in collaborative environments, which fits the company's emphasis on asynchronous collaboration and teamwork.

Gaps:
- Limited mention of TypeScript experience in the resume, which is a significant part of the job's te

In [19]:
import numpy as np
from sentence_transformers import SentenceTransformer

def compatibility_score(job_text: str, resume_text: str, model_name="all-MiniLM-L6-v2") -> float:
    model = SentenceTransformer(model_name)
    emb = model.encode([job_text, resume_text], convert_to_numpy=True).astype("float32")
    emb = emb / np.clip(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12, None)
    return float(np.dot(emb[0], emb[1]))  # cosine

score = compatibility_score(
    read_txt("../data/jobs/sample_job.txt"),
    read_txt("../data/resume_extract_text/55.txt")
)
score


0.6648896932601929

In [27]:
# =============================
# MODE 3 — PAIR COMPATIBILITY (RESUME ↔ JOB)
# =============================

import os
from agents.explainer_agent import explain_match_with_llm, build_llm_client
from utils.json_utils import safe_json_parse

client = build_llm_client()

def read_txt(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

RESUME_DIR = "../data/resume_extract_text"
JOB_DIR = "../data/jobs"

# Example inputs you choose (one resume + one job)
resume_filename = "55.txt"
job_filename = "sample_job.txt"

resume_pair_text = read_txt(resume_filename, RESUME_DIR)
job_pair_text = read_txt(job_filename, JOB_DIR)

# score must be computed before (cosine similarity between embeddings)
# score = ...
out = explain_match_with_llm(
    mode="pair_compatibility",
    similarity_score=score,
    resume_text=resume_pair_text,
    job_text=job_pair_text,
    client=client,
)

result = safe_json_parse(out["raw_json"])

print("=" * 90)
print(f"[PAIR COMPATIBILITY] Resume: {resume_filename} | Job: {job_filename} | Score: {score:.3f}\n")
print(result.get("explanation", ""))

print("\nStrengths:")
for s in result.get("strengths", []):
    print("-", s)

print("\nGaps:")
for g in result.get("gaps", []):
    print("-", g)

print("\nDecision:", result.get("decision", "N/A"))


[PAIR COMPATIBILITY] Resume: 55.txt | Job: sample_job.txt | Score: 0.665

The resume shows a solid foundation in frontend development, particularly with React and TypeScript, which aligns well with the job offer. However, while there is relevant experience, the candidate's background also includes full-stack development, which may not fully match the specific focus on frontend work emphasized in the job description. The similarity score of 0.665 indicates a moderate alignment between the candidate's skills and the job requirements.

Strengths:
- Proficient in React and TypeScript, matching the core technologies required for the role.
- Experience in frontend development with a variety of frameworks, showcasing versatility.
- Solid understanding of complex state management through Redux, which is crucial for the position.

Gaps:
- Limited specific experience with Tailwind CSS and Shadcn's UI components, which are part of the company's tech stack.
- The resume indicates a broader full-st

In [ ]:
# =========================================
# MODE 3 — PAIR COMPATIBILITY + CV IMPROVEMENT
# =========================================

import os
from agents.explainer_agent import explain_match_with_llm, build_llm_client
from utils.json_utils import safe_json_parse

client = build_llm_client()

def read_txt(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

RESUME_DIR = "../data/resume_extract_text"
JOB_DIR = "../data/jobs"

# Selected pair
resume_filename = "55.txt"
job_filename = "sample_job.txt"

resume_text = read_txt(resume_filename, RESUME_DIR)
job_text = read_txt(job_filename, JOB_DIR)

# similarity_score must already be computed via embeddings
# score = cosine_similarity(...)
# Example:
# score = 0.78

out = explain_match_with_llm(
    mode="pair_compatibility_with_improvement",
    similarity_score=score,
    resume_text=resume_text,
    job_text=job_text,
    client=client,
)

result = safe_json_parse(out["raw_json"])

# --------------------
# DISPLAY
# --------------------
print("=" * 90)
print(
    f"[PAIR COMPATIBILITY]\n"
    f"Resume: {resume_filename}\n"
    f"Job: {job_filename}\n"
    f"Matching score: {score:.3f}\n"
)

print("Explanation:")
print(result.get("explanation", ""))

print("\nStrengths:")
for s in result.get("strengths", []):
    print("-", s)

print("\nGaps:")
for g in result.get("gaps", []):
    print("-", g)

print("\nCV Improvement Suggestions (targeted for this job):")
for i, rec in enumerate(result.get("cv_improvements", []), start=1):
    print(f"{i}. {rec}")

print("\nFinal Decision:", result.get("decision", "N/A"))
